# 1b. Tensors, with pictures

This is the illustrated companion to [tensor-ladder.md](tensor-ladder.md), and the notebook
[01_the_cache.ipynb](01_the_cache.ipynb) §1.3 pointed here. Same ideas, now with pictures
instead of only tables: what a 1D, 2D, 3D and 4D tensor actually looks like, and why the
*same shape* can mean two completely different things depending on where in a pipeline you
are holding it.

Everything below is dummy data on purpose — small enough to draw completely. §6 repeats the
1D and 2D pictures with a real downloaded cache, so the dummy version and the real one sit
side by side.

In [ ]:
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import torch

torch.manual_seed(0)

## 1D: a vector — one item, one representation

The plainest case: a single row of numbers. Eight of them here, so the whole thing fits on
screen; a real cache row is the same picture, just 384 wide (§6).

In [ ]:
vector = torch.randn(8)  # one item, one representation

fig, ax = plt.subplots(figsize=(6, 1))
ax.imshow(vector.unsqueeze(0), cmap="viridis", aspect="auto")
ax.set_yticks([])
ax.set_xticks(range(8))
ax.set_title(f"1D tensor, shape {tuple(vector.shape)}")
plt.show()

## 2D: the same shape, two different pictures

Stack rows of the 1D vector above and you get a 2D tensor — but *which* 2D tensor depends
entirely on what each row is:

- **`(B, D)`** — a *batch*: five different items, each one an independent 1D vector.
- **`(C, D)`** — one *chunked* item: a single document (or image, or signal) split into
  five pieces, each piece a chunk's own vector.

Below are two `(5, 8)` tensors. Formally they are identical — same rank, same shape,
nothing in the object says which is which. Only the row labels (which you have to track
yourself, in variable names and comments) say what they mean.

In [ ]:
batch_of_items = torch.randn(5, 8)  # (B, D): 5 different items
chunks_of_item = torch.randn(5, 8)  # (C, D): 1 item, split into 5 chunks -- same shape!

fig, axes = plt.subplots(1, 2, figsize=(11, 2.4))
axes[0].imshow(batch_of_items, cmap="viridis", aspect="auto")
axes[0].set_ylabel("item #")
axes[0].set_title(f"(B, D) = {tuple(batch_of_items.shape)}\n5 different items")

axes[1].imshow(chunks_of_item, cmap="viridis", aspect="auto")
axes[1].set_ylabel("chunk #")
axes[1].set_title(f"(C, D) = {tuple(chunks_of_item.shape)}\n1 item, 5 chunks")
plt.tight_layout()
plt.show()

Two identical-looking heatmaps, two unrelated meanings. This is
[tensor-ladder.md](tensor-ladder.md) §4's point made visual: `torch` cannot distinguish
"a batch of 5 items" from "1 item in 5 chunks" — both are just `(5, 8)`. Whether you are
before or after collation is the only thing that decides which picture you are holding,
and nothing in the tensor itself records it.

## 3D: a stack of the 2D picture, two ways

Add a third axis and there are — again — two unrelated things it commonly means.

**Reading 1 — a literal image.** `(H, W, 3)`: height, width, three colour channels. This
one is not a metaphor; it really is three stacked 2D grids, one per channel.

In [ ]:
H, W = 40, 40
image = torch.rand(H, W, 3)  # a tiny random "photo": (H, W, 3)

fig, axes = plt.subplots(1, 4, figsize=(13, 3))
axes[0].imshow(image)
axes[0].set_title(f"(H, W, 3) = {tuple(image.shape)}\nall three channels together")
for i, channel in enumerate(["R", "G", "B"]):
    axes[i + 1].imshow(image[:, :, i], cmap="gray")
    axes[i + 1].set_title(f"channel {channel}\none (H, W) plane")
for ax in axes:
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()

**Reading 2 — a batch of chunked items.** `(B, C, D)`: the `(C, D)` chunked-item picture
from §2, now stacked `B` times into a batch. Nothing about this shape is a photo — it is
four documents, each split into 5 chunks of 8 features. Drawn the same way as the RGB
channels above (small multiples, one panel per batch element) on purpose: the *drawing
technique* for "a stack of 2D grids" is the same regardless of what the grids mean.

In [ ]:
batch_chunked = torch.randn(4, 5, 8)  # (B, C, D): 4 documents, each chunked into 5 pieces

fig, axes = plt.subplots(1, 4, figsize=(13, 2.4))
for b, ax in enumerate(axes):
    ax.imshow(batch_chunked[b], cmap="viridis", aspect="auto")
    ax.set_title(f"document {b}\n(C, D) = {tuple(batch_chunked[b].shape)}")
    ax.set_xticks([])
    ax.set_yticks([] if b else range(5))
    if b == 0:
        ax.set_ylabel("chunk #")
fig.suptitle(f"(B, C, D) = {tuple(batch_chunked.shape)} -- a batch, each panel a whole (C, D) item")
plt.tight_layout()
plt.show()

## 4D: a stack of the 3D picture

One more axis, one more "batch of the previous rung": `(B, H, W, 3)` is a batch of `B`
literal images — each one the rank-3 photo-cube from §3, now stacked. If a 3D tensor is a
cube of numbers, a 4D tensor here is a stack of those cubes, drawn two ways below: as
actual thumbnails, and as a literal stack-of-cards schematic so the "stack of cubes"
picture is explicit.

In [ ]:
batch_images = torch.rand(6, 24, 24, 3)  # (B, H, W, 3): a batch of 6 tiny RGB images

fig, axes = plt.subplots(1, 6, figsize=(13, 2.2))
for b, ax in enumerate(axes):
    ax.imshow(batch_images[b])
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(f"#{b}")
fig.suptitle(f"(B, H, W, 3) = {tuple(batch_images.shape)} -- a stack of the rank-3 cubes above")
plt.tight_layout()
plt.show()

In [ ]:
n_cubes, size, offset = 5, 1.6, 0.35

fig, ax = plt.subplots(figsize=(6, 4))
for i in range(n_cubes):
    d = i * offset
    ax.add_patch(
        mpatches.Rectangle(
            (d, d), size, size,
            facecolor=plt.cm.viridis(i / n_cubes),
            edgecolor="black", alpha=0.85, zorder=n_cubes - i,
        )
    )
ax.set_xlim(-0.5, n_cubes * offset + size + 0.5)
ax.set_ylim(-0.5, n_cubes * offset + size + 0.5)
ax.set_aspect("equal")
ax.axis("off")
ax.set_title(f"(B, H, W, 3): B={n_cubes} stacked (H, W, 3) cubes")
plt.show()

## Recap: rank, shape, and (at least) two meanings each

| Rank | Shape | What we drew | A second, unrelated reading |
|:---:|---|---|---|
| 1D | `(D)` | one item's representation | — (this is the ground floor) |
| 2D | `(B, D)` | a batch of items | `(C, D)`: one item, chunked |
| 3D | `(H, W, 3)` | an RGB image | `(B, C, D)`: a batch of chunked items |
| 4D | `(B, H, W, 3)` | a batch of images | — (rare in this course; see below) |

The pattern repeats at every rung: **going up one rank almost always means "a batch of
the previous rung"**, and the formal shape never tells you which semantic reading is
correct — that lives in your variable names, your comments, and (for a cache you did not
build) in `metadata.json`, exactly as [01_the_cache.ipynb](01_the_cache.ipynb) §1.2 said.

One practical note on 4D: this course rarely needs it directly. Raw images arrive as
`(B, H, W, 3)`, but `ImageVectorizer` collapses that whole cube straight to `(B, D)` —
which is the entire economic argument from notebook 01, seen from the shape side. You will
see 4D again only if you ever batch raw images by hand before encoding them.

## The dummy pictures, with real numbers

Same two shapes as §1 and §2 — a single vector, then a batch — pulled from an actual
downloaded cache instead of `torch.randn`. Compare the pictures side by side with the
dummy ones above: same kind of object, just wider and no longer random.

In [ ]:
from vectormesh import VectorCache

cache = VectorCache.from_hub("pttrn-io/eurosat-dinov2-small", split="test")
real_vector = cache["embed"][0]
print(f"one real cache row: {tuple(real_vector.shape)}")

fig, axes = plt.subplots(2, 1, figsize=(10, 2.6))
axes[0].imshow(vector.unsqueeze(0), cmap="viridis", aspect="auto")
axes[0].set_yticks([])
axes[0].set_title(f"dummy 1D tensor, shape {tuple(vector.shape)}")

axes[1].imshow(real_vector[:64].unsqueeze(0), cmap="viridis", aspect="auto")
axes[1].set_yticks([])
axes[1].set_title(f"real cache row -- first 64 of {real_vector.shape[0]}, same kind of object")
plt.tight_layout()
plt.show()

In [ ]:
from torch.utils.data import DataLoader
from vectormesh.data import Collate, OneHot

onehot = OneHot(num_classes=cache.num_classes, label_col="label", target_col="onehot")
collate = Collate(embedding_col="embed", target_col="onehot", padder=torch.stack)
loader = DataLoader(cache.dataset.map(onehot), batch_size=6, shuffle=True, collate_fn=collate)

X, y = next(iter(loader))
print(f"one batch: X {tuple(X.shape)}  <- (B, D), the section-2 picture again, for real vectors")

fig, ax = plt.subplots(figsize=(8, 2))
ax.imshow(X[:, :64], cmap="viridis", aspect="auto")
ax.set_ylabel("item #")
ax.set_xlabel("first 64 of 384 dims")
ax.set_title(f"(B, D) = {tuple(X.shape)} -- six real EuroSAT vectors, batched")
plt.show()

## Exercises

**E1 — three readings, one shape.** You are handed a tensor of shape `(32, 500, 3)` and
nothing else. List at least two plausible readings (e.g. a batch of small RGB thumbnails;
a batch of 3-sensor timeseries, 500 steps long; a batch of documents chunked into 500
pieces with a 3-number summary per chunk). For each, name one extra piece of information —
not the shape — that would let you tell it apart from the others.

**E2 — before a batch, chunks must agree.** Build three `(C, 8)` tensors by hand with
*different* `C` (say 3, 5, and 4 chunks). Try to `torch.stack` them into one `(B, C, D)`
batch directly, look at the error, and explain in one sentence why it fails. (You are not
expected to fix it here — `Collate` with a padder does this properly, and shows up in
notebook 03. This exercise is only about seeing *why* padding has to exist.)

**E3 — a different real cache.** Repeat the real-data cell above (§6) for
`pttrn-io/dtd-dinov2-small` instead of EuroSAT. Same shapes, different rows — confirm that
for yourself rather than taking it on faith.